Jon Ball<br>
4/21/2022

## Objective: using labeled data as a guide, create a single regex for identifying messages about academic outcomes
First, the proposed regex, which is *not* exhaustive:

In [ ]:
academic_filter = r"class|grade|point|averag|work|assignment|test|exam|quiz|math|science|english|report|passing|complet|quarter|semester|period"
print(academic_filter)

The words best suited for this task are nouns. "class" and "assignment," for example, have multiple word senses. But given the context of talking about school, it is likely that "class" and "assignment" are being used in the scholastic word sense. Some verbs, such as "fail," also have a limited meaning in this domain.
<br><br>
The regex above should be robust to situational context because it aligns with word usage in the broader domain of K-12 education. Each word in the regex has a clear "in school" meaning. Like the words that appear on a transcript, the regex consists of the vocabulary used in formal grading. It is not exhaustive; other subjects like *statistics* or *history* could be added (although they appear much less often than *math*).
<br><br>
Notably, the regex also omits words such as "course" that are used in common expressions (e.g., "of course") which would bias the regex.
<br><br>
The regex can function as a first-pass filter for identifying messsages about academic outcomes. 
<br><br>
It can then be used in conjunction with the following regex, which captures messages about `negative` academic outcomes, in a simple pos/neg classifiction scheme:

In [ ]:
neg_filter = r"negat|fail|not pass|absen|(not |hasn't )(been |be )?attend|\bmissing|\bmissed|not complet|incomplet|\blate\b"
print(neg_filter)

The regular expression above, which captures phrases like "currently failing class," or "absent yesterday," is very precise at identifying negative outcomes. This is because words like "fail" and "absent" are clear indications that a message is about a negative academic outcome. Normally, teachers do not use these words when students are performing well.
<br><br>
However, the word "late" is relatively more indeterminate; it has a communicative function ("sorry for the late reply"), in addition to the intended meaning of *late work* or *late to class*. The motivation for including it, however, is to capture a broader range of negative academic outcomes than just `fail|miss|absent`.
<br><br>
Lastly, concatenate the filters so we can calculate the conditional probability of `neg_filter` given `academic_filter` later:

In [ ]:
academic_filter += r"|" + neg_filter

### Validation:

In [ ]:
#import packages
import pandas as pd
import numpy as np
import spacy
import re

Read a sample of 100,000 text messages labeled 'academic' by odetta:

In [ ]:
acaDF = pd.read_csv('data/tpm_academic_100k.csv')
acaDF = acaDF[acaDF['TEXTENGLISH'].notnull()]
acaDF.info()

Create a function wrapper that tokenizes and lemmatizes messages:

In [ ]:
from nltk.tokenize import TweetTokenizer
twk = TweetTokenizer()
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

def lemmatize_(row):
    return [wnl.lemmatize(w) for w in twk.tokenize(row['TEXTENGLISH'])]

acaDF['TEXT'] = acaDF.apply(lemmatize_, axis=1)

In [ ]:
#Get word frequency counts
from collections import defaultdict
from nltk.corpus import stopwords
stop = stopwords.words('english')

def word_count(iterable):
    word_freq = defaultdict(int)
    total = 0
    
    for message in iterable:
        for token in message:
            if token not in stop:
                word_freq[token] += 1
                total += 1
                
    word_freq.update({k: v / total for k, v in word_freq.items()})
                
    return sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

In [ ]:
acad_freq = word_count(acaDF['TEXT'].tolist())
for tup in acad_freq[:200]:
    print(tup)

In [ ]:
#Load sample of 100k random messages
ranDF = pd.read_csv('data/tpm_random_100k.csv')
ranDF = ranDF[ranDF['TEXTENGLISH'].notnull()]
ranDF['TEXT'] = ranDF.apply(lemmatize_, axis=1)
ranDF.info()

In [ ]:
rand_freq = word_count(ranDF['TEXT'].tolist())
for tup in rand_freq[:200]:
    print(tup)

In [ ]:
def freq_ratio(acad_freq, rand_freq):
    log_ratios = []
    for tup1 in acad_freq:
        for tup2 in rand_freq:
            if tup1[0] == tup2[0]:
                ratio = np.log(tup1[1] / tup2[1])
                log_ratios.append((tup1[0], ratio))
    return sorted(log_ratios, key=lambda x: x[1], reverse=True)

In [ ]:
log_ratios = freq_ratio(acad_freq, rand_freq)
log_ratios[:100]

In [ ]:
academic_filter = r"class|course|grade|point|averag|work|assignment|test|exam|quiz|math|science|english|report|current|passing|complet|quarter|semester|period"

In [ ]:
neg_filter = r"fail|not pass|absen|(not |hasn't )(been |be )?attend|\bmissing|\bmissed|not complet|incomplet|\blate"

In [ ]:
academic_filter += "|" + neg_filter

In [ ]:
#Load sample of 100k teacher messages
testDF = pd.read_csv('data/rand100k_teachers.csv')
testDF = ranDF[ranDF['TEXTENGLISH'].notnull()]
testDF.info()

In [ ]:
testDF_filtered = testDF[testDF['TEXTENGLISH'].str.contains(academic_filter, regex=True, case=False)].copy()
x1 = testDF_filtered.shape[0]
x2 = testDF_filtered[testDF_filtered['TEXTENGLISH'].str.contains(neg_filter, regex=True, case=False)].shape[0]
print('Proportion of filtered academic messages which are labeled negative using this approach: {}'
      .format(round(x2 / x1, 2)))